# Checkpoint 1: data preparation and launch metrics

Execution edition of the original preparation and KPI workflow, using normalized local project inputs and the final participation/product snapshot. The sequence follows roster integration, sales/launch integration, product mapping, three-month metrics, coverage, and final joins.

Set `AWG_NORMALIZED_DIR` and `AWG_OUTPUT_DIR` as described in the execution guide. The original source is preserved in `archive/01_checkpoint1_data_and_metrics.ipynb`. Runtime corrections and metric rules are recorded in the notebook changes document.

## 1. Merge three files to a new one, named Participation:
## a. AIM
## b. One-Source
## c. Master

In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
from awg_analysis.data import load_data

INPUT_DIR = Path(os.environ["AWG_NORMALIZED_DIR"]).resolve()
OUTPUT_DIR = Path(os.environ["AWG_OUTPUT_DIR"]).resolve() / "checkpoint1"
if ROOT == OUTPUT_DIR or ROOT in OUTPUT_DIR.parents:
    raise ValueError("Choose an output directory outside the public repository")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
frames = load_data(INPUT_DIR)

In [ ]:
# Use the validated final participation snapshot for the executable edition.
df_Master = frames["stores"].rename(columns={"store_id": "Store_Code", "division": "Division Code"}).copy()
df_AIM = df_Master.loc[df_Master.program.str.startswith("AIM_"), ["Store_Code"]].assign(AIM=1)
df_OneSource = df_Master.loc[df_Master.program.str.contains("One Source"), ["Store_Code"]].assign(One_Source=1)

In [ ]:
# Preserve a one-row-per-store roster through both joins.
merged_Master_AIM = pd.merge(df_Master, df_AIM, on="Store_Code", how="left", validate="one_to_one")
Participation = pd.merge(merged_Master_AIM, df_OneSource, on="Store_Code", how="left", validate="one_to_one")
Participation[["AIM", "One_Source"]] = Participation[["AIM", "One_Source"]].fillna(0).astype(int)
Participation["program_family"] = "None"
Participation.loc[Participation.AIM.eq(1), "program_family"] = "AIM"
Participation.loc[Participation.One_Source.eq(1), "program_family"] = "One Source"
Participation.loc[Participation.AIM.eq(1) & Participation.One_Source.eq(1), "program_family"] = "Both"
Participation.loc[Participation.program.eq("Unknown"), "program_family"] = "Unknown"

In [ ]:
Participation.to_csv(OUTPUT_DIR / "Participation.csv", index=False)

## 2. Merge three files to a new one, named Sales_Data_Joined:
## a. the file with launch year/month, bsp, amap
## b. sales raw data
## c. Participation

In [ ]:
df_Sales = frames["sales"].rename(columns={
    "store_id": "Store_Code", "item_id": "Item_Code", "month": "Sales_Period",
    "units": "Billed Quantity", "sales": "Extended AWG Sell"}).copy()
df_Sales["Fiscal Year"] = df_Sales.Sales_Period.dt.year
df_Sales["Calendar Month Nbr"] = df_Sales.Sales_Period.dt.month

df_month = frames["items"].loc[frames["items"].launch_month.notna(), ["item_id", "launch_month"]].copy()
df_month = df_month.rename(columns={"item_id": "Item_Code", "launch_month": "Launch_Period"})
df_month["Launch_Year"] = df_month.Launch_Period.dt.year
df_month["Launch_Month"] = df_month.Launch_Period.dt.month
observed_months = set(frames["observation_months"].month)
df_month["Complete_3M"] = df_month.Launch_Period.map(lambda launch: all(launch + k in observed_months for k in (1, 2, 3)))
df_Participation = Participation.copy()

In [ ]:
merged_Sales_Participation = pd.merge(
    df_Sales, df_Participation, on="Store_Code", how="inner", validate="many_to_one")
merged_Sales_Participation_month = pd.merge(
    merged_Sales_Participation, df_month, on="Item_Code", how="inner", validate="many_to_one")

In [ ]:
merged_Sales_Participation_month.to_csv(OUTPUT_DIR / "Sales_Data_Joined.csv", index=False)

## 3. Merge two files to a new one, named Sales_Brands_Joined:
## a. Sales_Data_Joined
## b. Brands_Categories


In [ ]:
df_Sales_Joined = merged_Sales_Participation_month.copy()
df_Brands = frames["items"][["item_id", "brand", "category"]].rename(
    columns={"item_id": "Item_Code", "brand": "Brand", "category": "Categories"})

The executable edition uses the validated final product mapping and chunked many-to-one joins. The source archive retains the original intermediate-table workflow.

In [ ]:
# Canonical product keys are unique; process the fact table in chunks.
key = "Item_Code"
assert not df_Brands.duplicated(key).any()
df_Brands_unique = df_Brands.copy()
chunk_size = 50000
results = []
for start in range(0, len(df_Sales_Joined), chunk_size):
    chunk = df_Sales_Joined.iloc[start:start + chunk_size]
    results.append(chunk.merge(df_Brands_unique, on=key, how="inner", validate="many_to_one"))
Sales_Brands_Joined = pd.concat(results, ignore_index=True)
assert len(Sales_Brands_Joined) == len(df_Sales_Joined)

In [ ]:
Sales_Brands_Joined.to_csv(OUTPUT_DIR / "Sales_Brands_Joined.csv", index=False)

## 4. Calculate Metrics

### a. calculate the total sales and quantities for each store in 3 month after launched

In [ ]:
df = Sales_Brands_Joined.copy()

# The primary window uses L+1 through L+3 across year boundaries.
def select_complete_window(frame):
    sales_period = pd.PeriodIndex(frame["Sales_Period"].astype(str), freq="M")
    launch_period = pd.PeriodIndex(frame["Launch_Period"].astype(str), freq="M")
    offset = sales_period.astype("int64") - launch_period.astype("int64")
    complete = frame["Complete_3M"].astype(str).str.lower().eq("true")
    return frame.loc[complete & (offset >= 1) & (offset <= 3)].copy()

def make_window(launch):
    launch = pd.Period(launch, freq="M")
    return ",".join(str(launch + k) for k in (1, 2, 3))

df_3m = select_complete_window(df)
df_3m["Months Window"] = df_3m.Launch_Period.map(make_window)
result = (
    df_3m.groupby(["Store_Code", "Launch_Year", "Launch_Month", "Months Window"], as_index=False)
    .agg({"Billed Quantity": "sum", "Extended AWG Sell": "sum"})
    .rename(columns={"Billed Quantity": "Billed Quantity_3M", "Extended AWG Sell": "Extended AWG Sell_3M"})
)
result.to_csv(OUTPUT_DIR / "store_launch_3m.csv", index=False)
result.head()

### b. calculate distribution of stores which sell items in 3 months after items launched

In [ ]:
# Item coverage uses positive net monthly quantity and the declared roster.
df_active = df_3m.loc[df_3m["Billed Quantity"].gt(0)]
active = df_active.groupby("Item_Code").Store_Code.nunique().rename("Active Stores")
result2 = df_month.loc[df_month.Complete_3M].merge(active, on="Item_Code", how="left", validate="one_to_one")
result2["Active Stores"] = result2["Active Stores"].fillna(0).astype(int)
result2["Months Window"] = result2.Launch_Period.map(make_window)
result2["Total Stores"] = len(df_Master)
result2["%Stores_in_3M"] = 100 * result2["Active Stores"] / result2["Total Stores"]
result2.to_csv(OUTPUT_DIR / "Item_Store_Coverage_3M.csv", index=False)
result2.head()

## 5. Merge Sales_Brands_Joined & Item_Store_Coverage_3M_with_window, named Sales_Brands_3M

In [ ]:
# Coverage is item-level; validated item/cohort keys preserve the sales grain.
main = Sales_Brands_Joined.copy()
post = result2[["Item_Code", "Launch_Year", "Launch_Month", "Active Stores", "Total Stores", "%Stores_in_3M"]]
keys = ["Item_Code", "Launch_Year", "Launch_Month"]
merged = main.merge(post, on=keys, how="left", validate="many_to_one")
assert len(merged) == len(main)
merged.to_csv(OUTPUT_DIR / "Sales_Brands_3M.csv", index=False)

# Reconcile independently calculated totals with the shared implementation.
from awg_analysis.metrics import launch_metrics
reference = launch_metrics(frames)
assert abs(result["Extended AWG Sell_3M"].sum() - reference["store_3m"].sales_3m.sum()) < 1e-6
coverage_check = result2.merge(reference["item_coverage_3m"][["item_id", "coverage_pct"]],
                              left_on="Item_Code", right_on="item_id", validate="one_to_one")
assert (coverage_check["%Stores_in_3M"] - coverage_check.coverage_pct).abs().lt(1e-9).all()
print("Checkpoint 1 totals, coverage, and join grain reconciled.")